In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import tensorflow as tf
import numpy as np
import os
import zipfile
from PIL import Image
from io import BytesIO
from keras import ops
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Conv2D,LeakyReLU,SpectralNormalization,SpatialDropout2D,Input,Conv2DTranspose,Add,GroupNormalization,ReLU,RandomTranslation,RandomFlip,Lambda,UpSampling2D
from tensorflow.keras import callbacks,mixed_precision

#### *loading the tfrec images into my notebook*

In [4]:
import glob
monet_tfrec_directory= '/kaggle/input/competitions/gan-getting-started/monet_tfrec/*.tfrec'
photo_tfrec_directory= '/kaggle/input/competitions/gan-getting-started/photo_tfrec/*.tfrec'
monet_files= sorted(glob.glob(monet_tfrec_directory))
photo_files= sorted(glob.glob(photo_tfrec_directory))
print(f'number of tfrec containers in monet_tfrec: {len(monet_files)}')
print(f"number of tfrec containers in photo_tfrec: {len(photo_files)}")

number of tfrec containers in monet_tfrec: 5
number of tfrec containers in photo_tfrec: 20


In [5]:
image_feature_description={
    'target': tf.io.FixedLenFeature([], dtype=tf.string),
    'image': tf.io.FixedLenFeature([], dtype=tf.string)
}

def parse_image_function(ProtocolBuf_example):
    features= tf.io.parse_single_example(ProtocolBuf_example, image_feature_description)
    image_target= features['target']
    image_target= tf.reshape(image_target,[1])
    image= tf.io.decode_jpeg(features['image'], channels=3)
    image= tf.cast(image, tf.float32)
    image= (image/127.5)-1.0
    image= tf.reshape(image, [256,256,3])
    return image, image_target

def load_monet_dataset(filenames,batch_size):
    raw_dataset= tf.data.TFRecordDataset(filenames)
    parsed_dataset= raw_dataset.map(parse_image_function, num_parallel_calls= tf.data.AUTOTUNE)
    parsed_dataset= parsed_dataset.cache()
    parsed_dataset= parsed_dataset.shuffle(buffer_size=256)
    parsed_dataset= parsed_dataset.repeat()
    batch= parsed_dataset.batch(batch_size)
    batch= batch.prefetch(tf.data.AUTOTUNE)
    return batch

def load_real_photos(filenames,batch_size):
    raw_dataset= tf.data.TFRecordDataset(filenames)
    parsed_dataset= raw_dataset.map(parse_image_function, num_parallel_calls= tf.data.AUTOTUNE)
    parsed_dataset= parsed_dataset.shuffle(buffer_size=256)
    batch= parsed_dataset.batch(batch_size)
    batch= batch.prefetch(tf.data.AUTOTUNE)
    return batch

In [ ]:
monet_dataset= load_monet_dataset(monet_files,2)
photo_dataset= load_real_photos(photo_files,2)
final_dataset= tf.data.Dataset.zip(monet_dataset,photo_dataset)

#### *defining architecture of the Discriminator*

In [7]:
def discriminator(INPUT_SHAPE=(256,256,3)):
    model= Sequential()
    model.add(Conv2D(64,(4,4),padding='same',input_shape=INPUT_SHAPE))
    model.add(LeakyReLU(negative_slope=0.2))
    
    model.add(SpectralNormalization(
        Conv2D(128,(4,4),strides=(2,2),padding='same')))
    model.add(LeakyReLU(negative_slope=0.2))
    
    model.add(SpectralNormalization(
        Conv2D(256,(4,4),strides=(2,2),padding='same')))
    model.add(LeakyReLU(negative_slope=0.2))
    
    model.add(SpectralNormalization(
        Conv2D(512,(4,4),strides=(2,2),padding='same')))
    model.add(LeakyReLU(negative_slope=0.2))
    
    model.add(SpectralNormalization(
        Conv2D(1,(4,4),strides=(1,1),padding='same',dtype='float32')))
    
    return model

In [8]:
D_x= discriminator()
D_y= discriminator()
D_x.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 64)   │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 256, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectral_normalization          │ (None, 128, 128, 128)  │       131,328 │
│ (SpectralNormalization)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 128, 128, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectral_normalization_1        │ (None, 64, 64, 256)    │       524,800 │
│ (SpectralNormalization)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectral_normalization_2        │ (None, 32, 32, 512)    │     2,098,176 │
│ (SpectralNormalization)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 32, 32, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectral_normalization_3        │ (None, 32, 32, 1)      │         8,194 │
│ (SpectralNormalization)         │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,765,634 (10.55 MB)

 Trainable params: 2,764,737 (10.55 MB)

 Non-trainable params: 897 (3.50 KB)

#### *defining the architecture of the generator using keras functional API*

In [9]:
def generator(INPUT_SHAPE=(256,256,3)):
    inputs= Input(shape=INPUT_SHAPE)
    x= inputs
    x= Lambda(lambda t: tf.pad(t,[[0,0],[3,3],[3,3],[0,0]],mode='REFLECT'),
            output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 6, input_shape[2] + 6, input_shape[3]))(x)
    x= Conv2D(64,kernel_size=(7,7),strides=(1,1),padding='valid',use_bias=False)(x)
    x= GroupNormalization(groups=-1)(x)
    x= ReLU()(x)
    
    x= Lambda(lambda t: tf.pad(t,[[0,0],[1,1],[1,1],[0,0]],mode='REFLECT'),
             output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3]))(x)
    x= Conv2D(128,kernel_size=(3,3),strides=(2,2),padding='valid',use_bias=False)(x)
    x= GroupNormalization(groups=-1)(x)
    x= ReLU()(x)
    
    x= Lambda(lambda t: tf.pad(t,[[0,0],[1,1],[1,1],[0,0]],mode='REFLECT'),
             output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3]))(x)
    x= Conv2D(256,kernel_size=(3,3),strides=(2,2),padding='valid',use_bias=False)(x)
    x= GroupNormalization(groups=-1)(x)
    x= ReLU()(x)

    for _ in range(9):
        residual=x
        f_x= Lambda(lambda t: tf.pad(t,[[0,0],[1,1],[1,1],[0,0]],mode='REFLECT'),
                   output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3]))(x)
        f_x= Conv2D(256,kernel_size=(3,3),padding='valid',use_bias=False)(f_x)
        f_x= GroupNormalization(groups=-1)(f_x)
        f_x= ReLU()(f_x)
        
        f_x= Lambda(lambda t: tf.pad(t,[[0,0],[1,1],[1,1],[0,0]],mode='REFLECT'),
                   output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3]))(f_x)
        f_x= Conv2D(256,kernel_size=(3,3),padding='valid',use_bias=False)(f_x)
        f_x= GroupNormalization(groups=-1)(f_x)

        x= Add()([residual,f_x])
        
    x= UpSampling2D(size=(2,2),interpolation='bilinear')(x)
    x = Lambda(lambda t: tf.pad(t, [[0,0],[1,1],[1,1],[0,0]], mode='REFLECT'),
              output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3]))(x)
    x= Conv2D(128,kernel_size=(3,3),padding='valid',use_bias=False)(x)
    x= GroupNormalization(groups=-1)(x)
    x= ReLU()(x)
    
    x= UpSampling2D(size=(2,2),interpolation='bilinear')(x)
    x = Lambda(lambda t: tf.pad(t, [[0,0],[1,1],[1,1],[0,0]], mode='REFLECT'),
              output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 2, input_shape[2] + 2, input_shape[3]))(x)
    x= Conv2D(64,kernel_size=(3,3),padding='valid',use_bias=False)(x)
    x= GroupNormalization(groups=-1)(x)
    x= ReLU()(x) 

    x = Lambda(lambda t: tf.pad(t, [[0,0],[3,3],[3,3],[0,0]], mode='REFLECT'),
              output_shape=lambda input_shape: (input_shape[0], input_shape[1] + 6, input_shape[2] + 6, input_shape[3]))(x)
    outputs= Conv2D(3,kernel_size=(7,7),strides=1,padding='valid',activation='tanh',dtype='float32')(x)
    generator_model= Model(inputs=inputs, outputs=outputs, name='Resnet')
    return generator_model

In [10]:
G_A2B= generator()
G_B2A= generator()
G_A2B.summary()

Model: "Resnet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 262, 262,  │          0 │ input_layer_2[0]… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 256, 256,  │      9,408 │ lambda[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalization │ (None, 256, 256,  │        128 │ conv2d_10[0][0]   │
│ (GroupNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 256, 256,  │          0 │ group_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 258, 258,  │          0 │ re_lu[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 128, 128,  │     73,728 │ lambda_1[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_11[0][0]   │
│ (GroupNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 128, 128,  │          0 │ group_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 130, 130,  │          0 │ re_lu_1[0][0]     │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 64, 64,    │    294,912 │ lambda_2[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_12[0][0]   │
│ (GroupNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 64, 64,    │          0 │ group_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 66, 66,    │          0 │ re_lu_2[0][0]     │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 64, 64,    │    589,824 │ lambda_3[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_13[0][0]   │
│ (GroupNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 64, 64,    │          0 │ group_normalizat

 Total params: 11,383,427 (43.42 MB)

 Trainable params: 11,383,427 (43.42 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
class MonetGAN(Model):
    def __init__(self,G_A2B,G_B2A,D_x,D_y,lambda_cyc,lambda_id):
        super(MonetGAN,self).__init__()
        self.G_A2B= G_A2B
        self.G_B2A= G_B2A
        self.D_x= D_x
        self.D_y= D_y
        self.lambda_cyc= lambda_cyc
        self.lambda_id= lambda_id
        self.g_loss_tracker = tf.keras.metrics.Mean(name="g_loss")
        self.d_x_loss_tracker = tf.keras.metrics.Mean(name="D_x_loss")
        self.d_y_loss_tracker = tf.keras.metrics.Mean(name="D_y_loss")
        self.spatial_augment= Sequential([
            RandomFlip('horizontal'),
            RandomTranslation(
                height_factor= 0.08,
                width_factor= 0.15,
                fill_mode='reflect',
                interpolation='bilinear'
            )
        ])

    def compile(self,adv_loss,id_loss,cyc_loss,G_optimizer,D_x_optimizer,D_y_optimizer):
        super(MonetGAN,self).compile()
        self.adv_loss= adv_loss
        self.id_loss= id_loss
        self.cyc_loss= cyc_loss
        self.G_optimizer= G_optimizer
        self.D_x_optimizer= D_x_optimizer
        self.D_y_optimizer= D_y_optimizer

    @tf.function
    def train_step(self,batch_data):
        monet_tuple,photo_tuple= batch_data
        real_monet,_= monet_tuple
        real_photo,_= photo_tuple
        
        with tf.GradientTape() as tape:
            id_photo= self.G_B2A(real_photo,training=True)
            id_monet= self.G_A2B(real_monet,training=True)
            id_photo_loss= self.id_loss(real_photo,id_photo)*(self.lambda_id)
            id_monet_loss= self.id_loss(real_monet,id_monet)*(self.lambda_id)

            fake_monet= self.G_A2B(real_photo,training=True)
            reconstructed_photo= self.G_B2A(fake_monet,training=True)
            fake_photo= self.G_B2A(real_monet,training=True)
            reconstructed_monet= self.G_A2B(fake_photo,training=True)
            cycle_photo_loss= self.cyc_loss(real_photo,reconstructed_photo)*(self.lambda_cyc)
            cycle_monet_loss= self.cyc_loss(real_monet,reconstructed_monet)*(self.lambda_cyc)

            augmented_real_monet= self.spatial_augment(real_monet,training=True)
            augmented_fake_monet= self.spatial_augment(fake_monet,training=True)
            augmented_real_photo= self.spatial_augment(real_photo,training=True)
            augmented_fake_photo= self.spatial_augment(fake_photo,training=True)
            fake_monet_preds= self.D_x(augmented_fake_monet,training=True)
            fake_photo_preds= self.D_y(augmented_fake_photo,training=True)
            adv_monet_loss= self.adv_loss(tf.ones_like(fake_monet_preds),fake_monet_preds)
            adv_photo_loss= self.adv_loss(tf.ones_like(fake_photo_preds),fake_photo_preds)

            total_generator_loss= id_photo_loss + id_monet_loss + cycle_photo_loss + cycle_monet_loss + adv_monet_loss + adv_photo_loss
            generator_params= self.G_A2B.trainable_variables + self.G_B2A.trainable_variables
            generator_gradients = tape.gradient(total_generator_loss,generator_params)
            self.G_optimizer.apply_gradients(zip(generator_gradients,generator_params))

        with tf.GradientTape() as tape:
            real_monet_preds= self.D_x(augmented_real_monet,training=True)
            fake_monet_preds= self.D_x(augmented_fake_monet,training=True)
            adv_monet_real_loss= self.adv_loss(tf.ones_like(real_monet_preds),real_monet_preds)
            adv_monet_fake_loss= self.adv_loss(tf.zeros_like(fake_monet_preds),fake_monet_preds)
            total_D_x_loss= (adv_monet_real_loss + adv_monet_fake_loss)*0.5
            D_x_gradients = tape.gradient(total_D_x_loss,self.D_x.trainable_variables)
            self.D_x_optimizer.apply_gradients(zip(D_x_gradients,self.D_x.trainable_variables))

        with tf.GradientTape() as tape:
            real_photo_preds= self.D_y(augmented_real_photo,training=True)
            fake_photo_preds= self.D_y(augmented_fake_photo,training=True)
            adv_photo_real_loss= self.adv_loss(tf.ones_like(real_photo_preds),real_photo_preds)
            adv_photo_fake_loss= self.adv_loss(tf.zeros_like(fake_photo_preds),fake_photo_preds)
            total_D_y_loss= (adv_photo_real_loss + adv_photo_fake_loss)*0.5
            D_y_gradients = tape.gradient(total_D_y_loss,self.D_y.trainable_variables)
            self.D_y_optimizer.apply_gradients(zip(D_y_gradients,self.D_y.trainable_variables))
        
        self.g_loss_tracker.update_state(total_generator_loss)
        self.d_x_loss_tracker.update_state(total_D_x_loss)
        self.d_y_loss_tracker.update_state(total_D_y_loss)
        return{
            'g_loss': self.g_loss_tracker.result(),
            'D_x_loss': self.d_x_loss_tracker.result(),
            'D_y_loss': self.d_y_loss_tracker.result()
        }
    @property
    def metrics(self):
        return [self.g_loss_tracker, self.d_x_loss_tracker, self.d_y_loss_tracker]

#### *defining a decay function for the learning rate to get better results- the learning rate drops from 0.0002 to 0.00002 after the 50th epoch*

In [19]:
def lr_scheduler(epoch,lr):
    if epoch<30:
        return 2e-4
    else:
        return 2e-4 - (((2e-4 - 2e-5)/30)*(epoch-30))
lr_callback= tf.keras.callbacks.LearningRateScheduler(lr_scheduler)

In [20]:
monetGAN= MonetGAN(
    G_A2B= G_A2B,
    G_B2A= G_B2A,
    D_x= D_x,
    D_y= D_y,
    lambda_cyc= 10.0,
    lambda_id= 2.5
    
    )

monetGAN.compile(
    adv_loss= tf.keras.losses.MeanSquaredError(),
    id_loss= tf.keras.losses.MeanAbsoluteError(),
    cyc_loss= tf.keras.losses.MeanAbsoluteError(),
    G_optimizer= tf.keras.optimizers.Adam(learning_rate=2e-4,beta_1=0.5),
    D_x_optimizer= tf.keras.optimizers.Adam(learning_rate=2e-4,beta_1=0.5),
    D_y_optimizer= tf.keras.optimizers.Adam(learning_rate=2e-4,beta_1=0.5)
    )

callbacks=[lr_callback]

In [ ]:
history= monetGAN.fit(
    final_dataset,
    epochs=60,
    steps_per_epoch=1000,
    callbacks= callbacks,
    verbose=1)

#### *saving the weights of the model after training*

In [ ]:
monetGAN.save_weights('/kaggle/working/final_model.weights.h5')

#### *plotting the generator loss function: 'g_loss' and the discriminator loss functions: 'D_x_loss' and 'D_y_loss'*

In [ ]:
epochs_range = range(1, len(history.history["g_loss"]) + 1)

plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["g_loss"], label="Generator Loss", color="royalblue", linewidth=2)
plt.title("Total Generator Loss Over Epochs", fontsize=14, fontweight="bold")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss Value", fontsize=12)
plt.grid(True, alpha=0.6)
plt.legend(fontsize=11)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["D_x_loss"], label="D_x Loss (Monet)", color="crimson", linewidth=2)
plt.plot(epochs_range, history.history["D_y_loss"], label="D_y Loss (Photo)", color="darkorange", linewidth=2)
plt.title("Discriminator Losses Over Epochs", fontsize=14, fontweight="bold")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss Value", fontsize=12)
plt.grid(True, alpha=0.6)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

#### *saving the images into 'monet_generated' temporarily and then using zipfile library to copy the images from the monet_generated directory and save it as images.zip*

In [ ]:
zip_path= '/kaggle/working/images.zip'
print('starting to store image into the file')

total_images=7038
image_counter=0
batch_size=2
total_steps= total_images // batch_size

@tf.function
def generate_monet(img):
    return monetGAN.G_A2B(img,training=False)

with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zip_file:
    for photo in tqdm(photo_dataset,desc='generating and zipping',total=total_steps,miniters=10):
        if isinstance(photo,tuple):
            img_tensor,_= photo
        else:
            img_tensor= photo
        generated_monet= generate_monet(img_tensor)
        generated_monet= keras.ops.convert_to_numpy(generated_monet)
        for i in range(generated_monet.shape[0]):
            single_img= generated_monet[i]
            single_img= (single_img*127.5)+127.5
            single_img= np.clip(single_img,0,255).astype(np.uint8)

            img= Image.fromarray(single_img)
            buffer= BytesIO()
            img.save(buffer,'JPEG',quality=95)
            zip_file.writestr(f'monet_{image_counter:04d}.jpg',buffer.getvalue())
            image_counter+=1
            buffer.close()
print(f'\nall images successfully saved inside {zip_path}')

#### *displaying 10 images to manually evaluate the performance of the cycleGAN*

In [ ]:
samples= photo_dataset.take(10)
plt.figure(figsize=(10,25))
idx=1
for photo in samples:
    if isinstance(photo,tuple):
        img_tensor,_= photo
    else:
        img_tensor= photo
    generated_monet= generate_monet(img_tensor)
    generated_monet= generated_monet[0].numpy()
    generated_monet= ((generated_monet*127.5) + 127.5)
    generated_monet= np.clip(generated_monet,0,255).astype(np.uint8)
    actual_photo= img_tensor[0].numpy()
    actual_photo= (actual_photo*127.5 + 127.5)
    actual_photo= np.clip(actual_photo,0,255).astype(np.uint8)
    
    plt.subplot(10, 2, idx)
    plt.imshow(actual_photo)
    plt.title("Original Photo Input", fontsize=10, fontweight="bold")
    plt.axis("off")
    
    
    plt.subplot(10, 2, idx + 1)
    plt.imshow(generated_monet)
    plt.title("Generated Monet Style", fontsize=10, fontweight="bold")
    plt.axis("off")
    
    idx += 2

plt.tight_layout()
plt.show()